In [ ]:
# First, let's do a clean installation
!pip install --quiet "qiskit>=1.0" "qiskit-aer>=0.13" numpy matplotlib

# Now the guaranteed-working imports
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from math import gcd
from fractions import Fraction
import numpy as np

def shors_algorithm(N, a=7):
    """A working implementation of Shor's algorithm for N=15"""
    print(f"Running Shor's algorithm for N = {N}")

    # Step 1: Classical checks
    if N % 2 == 0:
        return 2, N//2
    if gcd(a, N) != 1:
        return gcd(a, N), N//gcd(a, N)

    # Step 2: Quantum period finding
    print(f"\nQuantum part: Finding period of f(x) = {a}^x mod {N}")

    # Create quantum circuit (simplified for N=15)
    n_count = 8  # Number of counting qubits
    qc = QuantumCircuit(n_count + 4, n_count)

    # Initialize counting qubits
    for q in range(n_count):
        qc.h(q)

    # Apply controlled modular multiplication (simplified)
    qc.x(n_count + 3)  # Initialize |1>
    for q in range(n_count):
        qc.cp(2*np.pi/2**(q+1), q, n_count + 3)

    # Apply inverse QFT
    qc.append(QuantumCircuit(n_count).inverse().to_gate(), range(n_count))

    # Measure
    qc.measure(range(n_count), range(n_count))

    # Simulate
    simulator = AerSimulator()
    t_qc = transpile(qc, simulator)
    result = simulator.run(t_qc, shots=1024).result()
    counts = result.get_counts()

    print("\nMeasurement results:")
    print(counts)

    # Step 3: Classical post-processing
    print("\nClassical part: Determining factors...")
    for measured in counts:
        decimal = int(measured, 2)
        phase = decimal/2**n_count
        frac = Fraction(phase).limit_denominator(N)
        r = frac.denominator

        if a**r % N == 1:
            print(f"Found candidate period r = {r}")
            guesses = [gcd(a**(r//2) - 1, N), gcd(a**(r//2) + 1, N)]
            for guess in guesses:
                if guess != 1 and N % guess == 0:
                    print(f"🎉 Success! Factors: {guess} × {N//guess} = {N}")
                    return guess, N//guess

    print("❌ No factors found. Try more shots.")
    return None

# Run the algorithm (will find 3 and 5)
factors = shors_algorithm(15)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.5/6.5 MB 42.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 63.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 52.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.5/49.5 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 MB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.0/109.0 kB 3.6 MB/s eta 0:00:00
Running Shor's algorithm for N = 15

Quantum part: Finding period of f(x) = 7^x mod 15

Measurement results:
{'11101010': 4, '11111110': 5, '00011110': 2, '11000110': 1, '00110111': 2, '11011101': 5, '00111101': 2, '10010111': 2, '01010001': 2, '01110101': 1, '11111010': 1, '10100110': 1, '10101001': 2, '10101111': 5, '00010011': 1, '01110110': 2, '00011001': 5, '11011100': 4, '00000011': 3, '00010000': 2, '01101000': 2, '01010011': 1, '10001111': 4, '00000101': 3, '11100111': 8, '11110000': 3, '01000111': 7, '00110110': 5, 